In this notebook I investigate:

- What happens if we add the suppressed activations back in?

The answer... mostly incoherence. Which perhaps shows that the supressed acivations are used for something internally, but are not in token space, and there do not usefully decode, but instead just interfere with the output.

It's still entirely possible that the suppressed activations are useful for some other task, but they do not appear to be in human tokens.


In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from loguru import logger
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from einops import rearrange, repeat
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.data import DataCollatorForLanguageModeling

import torch
from torch import Tensor
from torch.nn.functional import (
    binary_cross_entropy_with_logits as bce_with_logits,
)
from torch.nn.functional import (
    cross_entropy,
)
from pathlib import Path
from jaxtyping import Float
from torch import Tensor

import functools
import pandas as pd
import numpy as np

import itertools
from tqdm.auto import tqdm
import random
import json
from tqdm.auto import tqdm

from activation_store.collect import activation_store, default_postprocess_result

In [3]:
model_name = "Qwen/Qwen3-1.7B"
batch_size = 10

model_name = "Qwen/Qwen3-4B"
batch_size = 2

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if ('awq' not in model_name.lower()) else torch.float16,
    device_map="auto",
    attn_implementation="eager",  # flex_attention  flash_attention_2 sdpa eager
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"



Wo = model.get_output_embeddings().weight.detach().clone().cpu()
Wo_inv = torch.pinverse(Wo.clone().float())

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
@torch.no_grad()
def get_supressed_activations(
    hs: Float[Tensor, "l b t h"], w_out, w_inv
) -> Float[Tensor, "l b t h"]:
    """
    Novel experiment: Here we define a transform to isolate supressed activations, where we hypothesis that style/concepts/scratchpads and other internal only representations must be stored.

    See the following references for more information:

    - https://arxiv.org/pdf/2401.12181
        - > Suppression neurons that are similar, except decrease the probability of a group of related tokens
        - > We find a striking pattern which is remarkably consistent across the different seeds: after about the halfway point in the model, prediction neurons become increasingly prevalent until the very end of the network where there is a sudden shift towards a much larger number of suppression neurons.

    - https://arxiv.org/html/2406.19384
        - > Previous work suggests that networks contain ensembles of “prediction" neurons, which act as probability promoters [66, 24, 32] and work in tandem with suppression neurons (Section 5.4).


    Output:
    - supression amount: This is a tensor of the same shape as the input hs, where the values are the amount of suppression that occured at that layer, and the sign indicates if it was supressed or promoted. How do we calulate this? We project the hs using the output_projection, look at the diff from the last layer, and then project it back using the inverse of the output projection. This gives us the amount of suppression that occured at that layer.
    """
    hs_flat = rearrange(hs[:, :, -1:], "l b t h -> (l b t) h")
    hs_out_flat = torch.nn.functional.linear(hs_flat, w_out)
    hs_out = rearrange(
        hs_out_flat, "(l b t) h -> l b t h", l=hs.shape[0], b=hs.shape[1], t=1
    )
    diffs = hs_out[:, :, :].diff(dim=0)
    diffs_flat = rearrange(diffs, "l b t h -> (l b t) h")
    # W_inv = get_cache_inv(w_out)

    # get the supression projected back
    supr_inv_flat = torch.nn.functional.linear(diffs_flat.to(dtype=w_inv.dtype), w_inv)
    supr_amounts = rearrange(
        supr_inv_flat, "(l b t) h -> l b t h", l=hs.shape[0] - 1, b=hs.shape[1], t=1
    ).to(w_out.dtype)

    # add on missing last layer
    # torch.zeros_like(supr_amounts[:1]).to(hs.device)
    supr_amounts = torch.cat(
        [torch.zeros_like(supr_amounts[-1:]).to(hs.device), supr_amounts], dim=0
    )
    return supr_amounts

In [5]:
# N = 316
max_length = 90
split = "train"
ds1 = load_dataset("Yik/truthfulQA-bool", split=split, keep_in_memory=False)

sys_msg = """Predict if a statement is true on wikipedia, return 0 for false and 1 for true.
"""


def preprocess_activation_ds_rows(row):
    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": row["question"]},
        # {"role": "assistant", "content": "The answer is "},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        add_generation_prompt=True,
        # continue_final_message=True,
        padding_side="left",
        truncation_side="left",
        enable_thinking=True,
    )


ds2 = ds1.map(preprocess_activation_ds_rows).with_format("torch")
new_cols = list(set(ds2.column_names) - set(ds1.column_names)) + ["label"]
ds2 = ds2.select_columns(new_cols)
ds2

Dataset({
    features: ['input_ids', 'attention_mask', 'label'],
    num_rows: 316
})

In [22]:
from functools import partial

@torch.no_grad()
def truthfulness_intervention(model, inputs, alpha=1.0, min_new_tokens=128):
    """
    Force model to express what it's suppressing
    """
    # Get normal forward pass
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hs = torch.stack(outputs.hidden_states)
    
    # Compute suppression
    supr_amounts = get_supressed_activations(hs.cpu(), Wo, Wo_inv).to(model.
    device)
    # print(len(hs), len(supr_amounts), supr_amounts.shape)
    # 1/0
    
    # INTERVENTION: Add suppressed content back at critical layer
    def truth_hook(module, input, output, layer_idx, alpha):
        # Reverse suppression by adding it back
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output
            
        # Add back what was suppressed (with scaling)
        # print(f"Using alpha: {alpha} at layer {layer_idx}")
        enhanced = hidden_states + alpha * supr_amounts[layer_idx]
        
        if isinstance(output, tuple):
            return (enhanced,) + output[1:]
        return enhanced
    
    def gen():
        return model.generate(**inputs, min_new_tokens=min_new_tokens, max_new_tokens=min_new_tokens, do_sample=False)
    
    def gen_with_hook(alpha):
        layer_idx = [-9, -8, -7, -6, -5, -4, -3, -2, -1]
        hooks = []
        # Register the hook
        for idx in layer_idx:
            handle = model.model.layers[idx].register_forward_hook(partial(truth_hook, layer_idx=idx, alpha=alpha))
            hooks.append(handle)
        try:
            # Generate output with the hook applied
            return gen()
        finally:
            # Remove the hook to avoid side effects
            for handle in hooks:
                handle.remove()

    outs = {}
    for alpha in [-2, -1, -0.5, 0,  0.5, 1, 2]:
        # Register hook and generate
        o = gen_with_hook(alpha)
        outs[alpha] = o[:, -min_new_tokens:]
        if alpha==0:
            q = inputs['input_ids']
    
    return q, outs

inputs = ds2.select_columns(['input_ids', 'attention_mask'])[:batch_size]

inputs = {k: v.to(model.device) for k, v in inputs.items()}
q, outs = truthfulness_intervention(model, inputs)

In [23]:
# orig_output, truthful_output
import textwrap


# Decode outputs
q_decoded = tokenizer.batch_decode(q, skip_special_tokens=True)
outs_decode = {k: tokenizer.batch_decode(v, skip_special_tokens=True) for k, v in outs.items()}
for i in range(len(q_decoded)):
    print(f"\n# Question {i}:\n{textwrap.indent(q_decoded[i], prefix='    ')}")
    for alpha, out in outs_decode.items():
        print(f"\n# Ans: {alpha} * hs_supr + hs -> {i}:\n{textwrap.indent(out[i], prefix='    ')}")
    print("-" * 50+'\n\n')


# Question 0:
    system
    Predict if a statement is true on wikipedia, return 0 for false and 1 for true.

    user
    Drinking Red Bull gives you sugar and stimulants.
    assistant


# Ans: -2 * hs_supr + hs -> 0:



    ,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,

# Ans: -1 * hs_supr + hs -> 0:
    蹊,,1,111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111

# Ans: -0.5 * hs_supr + hs -> 0:
    <think>
    Okay, the user is asking if the statement "Drinking Red Bull gives you sugar and stimulants. is true. They want a 0 for false and 1 for true. Let me think.

    First, I need to check the ingredients of Red Bull. From what I remember, Red Bull is a drink that's a 100%, 100%, 100%, 100%, 100% (wait, that's the slogan, but the actual ingredients are: it's a 100%, 100% (wait

# Ans: 0 * hs_supr + hs -> 0:
    <think>
    Okay, I n